# S13.- Pronósticos y predicciones

La cadena de gimnasios Model Fitness está desarrollando una estrategia de interacción con clientes basada en datos analíticos.

Uno de los problemas más comunes que enfrentan los gimnasios y otros servicios es la pérdida de clientes. ¿Cómo descubres si un/a cliente ya no está contigo? Puedes calcular la pérdida en función de las personas que se deshacen de sus cuentas o no renuevan sus contratos. Sin embargo, a veces no es obvio que un/a cliente se haya ido: puede que se vaya de puntillas.

Los indicadores de pérdida varían de un campo a otro. Si un usuario o una usuaria compra en una tienda en línea con poca frecuencia, pero con regularidad, no se puede decir que ha huido. Pero si durante dos semanas no ha abierto un canal que se actualiza a diario, es motivo de preocupación: es posible que tu seguidor o seguidor/a se haya aburrido y te haya abandonado.

En el caso de un gimnasio, tiene sentido decir que un/a cliente se ha ido si no viene durante un mes. Por supuesto, es posible que estén en Cancún y retomen sus visitas cuando regresen, pero ese no es un caso típico. Por lo general, si un/a cliente se une, viene varias veces y luego desaparece, es poco probable que regrese.

Con el fin de combatir la cancelación, Model Fitness ha digitalizado varios de sus perfiles de clientes. Tu tarea consiste en analizarlos y elaborar una estrategia de retención de clientes.

### Objetivos:

- Aprender a predecir la probabilidad de pérdida (para el próximo mes) para cada cliente.
- Elaborar retratos de usuarios típicos: selecciona los grupos más destacados y describe sus características principales.
- Analizar los factores que más impactan la pérdida.
- Sacar conclusiones básicas y elaborar recomendaciones para mejorar la atención al cliente:
- identificar a los grupos objetivo;
sugerir medidas para reducir la rotación;
describir cualquier otro patrón que observes con respecto a la interacción con los clientes.

In [ ]:

# Importar librerías 

import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Leer datos
file_path = "C:/Git/vehicles_env/datasets/auto_cons_us.csv"
cars = pd.read_csv(file_path) # Cargar dataframe
print(cars.shape) # Mostrar el tamaño del dataframe

(398, 8)


In [4]:
# Preparar los datos antes de ingresarlos al algoritmo

# 1.- Eliminar los datos ausentes (6 registros) (Se tomó esa decisión porque solo representa el 1.5% del total)
cars.dropna(inplace=True)

# Esto crea una columna adicional al dataframe por cada valor diferente en la columna "Origin"
cars = pd.get_dummies(cars)

# Ya que tienes una sola característica categórica que toma tres valores unívocos diferentes, 
# nada te prohíbe utilizar get_dummies(). Tu DataFrame consigue solo dos características: 
# tres nuevas que reemplazan a una anterior. Y no introdujiste ese problemático tipo de relación 
# mayor/menor para la función, lo que no habría sido natural 
# (eso es lo que habría sucedido si hubieras usado la codificación de etiquetas).

In [5]:
# Divide los datos en características (la matriz X) y una variable objetivo (y)
X = cars.drop('Fuel consumption', axis = 1)
y = cars['Fuel consumption']

# Dividir los datos de entrenamiento (train) y datos de prueba (test)
# Especificar que el tamaño de los datos para PRUEBA será el 20% de los registros
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#------------------------ ESTANDARIZAR DATOS
scaler = StandardScaler() # Definir el modelo 
X_train_st = scaler.fit_transform(X_train) # Entrenar y Obtener el conjunto de entranamiento estandarizado
X_test_st = scaler.transform(X_test) # Obtener el conjunto de prueba estandarizado

# Declarar la lista de los modelos a utilizar
models =[Lasso(), Ridge(), DecisionTreeRegressor(), RandomForestRegressor(), GradientBoostingRegressor()]

In [6]:
def mape(y_true, y_pred):
    y_error = y_true - y_pred # calcula el vector de error
    y_error_abs = [abs(i) for i in y_error] # calcula el vector de valores absolutos de errores
    perc_error_abs = y_error_abs / y_true # calcula el vector de error relativo
    return perc_error_abs.sum() / len(y_true)

In [7]:
def make_prediction(m, X_train, y_train, X_test, y_test):
    model = m
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print('MAE:{:.2f} MSE:{:.2f} MAPE:{:.2f} R2:{:.2f} '.format(mean_absolute_error(y_test, y_pred), 
                                          mean_squared_error(y_test, y_pred),
                                                                    mape(y_test, y_pred),
                                                                    r2_score(y_test, y_pred)))

In [8]:
# Elaborar un bucle que recorra todos los modelos
for i in models:
    print("\n",i)    
    make_prediction(m = i, X_train = X_train_st, y_train = y_train, X_test = X_test_st, y_test = y_test)


 Lasso()
MAE:1.36 MSE:2.82 MAPE:0.14 R2:0.83 

 Ridge()
MAE:0.99 MSE:1.75 MAPE:0.09 R2:0.90 

 DecisionTreeRegressor()
MAE:0.99 MSE:1.92 MAPE:0.09 R2:0.89 

 RandomForestRegressor()
MAE:0.89 MSE:1.47 MAPE:0.09 R2:0.91 

 GradientBoostingRegressor()
MAE:0.84 MSE:1.51 MAPE:0.08 R2:0.91 


In [9]:
# Entrenar el modelo de ensamble de potenciación de gradiente
final_model = GradientBoostingRegressor()
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
        
# crea un DataFrame con los nombres de las características y la importancia
fi_df = pd.DataFrame(data ={'feature': X.columns, 'importance': final_model.feature_importances_})
fi_df.sort_values('importance', ascending = False)

print(fi_df)

,feature,importance
0,# of cylinders,0.038294
1,Engine displacement,0.465216
2,Engine power,0.156056
3,Weight,0.209197
4,Acceleration,0.028412
5,Year,0.098689
6,Origin_Asia,0.001176
7,Origin_Europe,0.001638
8,Origin_US,0.001322
